El presente notebook muestra los resultados de la implementación de un model141414o de redes neruonales convolucionales donde se emplean filtros de 3x3 y 5x5 en cada capa y se concatenan las salida para la entrada a la capa siguiente

In [2]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, ConvLSTM2D, Conv2D, TimeDistributed,
    Add, Activation, BatchNormalization, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.metrics import RootMeanSquaredError, MeanAbsolutePercentageError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt


In [8]:
# =====================================================
# CONFIGURACIÓN DE RUTAS
# =====================================================

# Rutas de tus imágenes - AJUSTA ESTAS RUTAS SEGÚN TU SISTEMA
d_original = '/home/fisica/bandas/banda13/ene_01_mes'  # Directorio con imágenes originales
d_reducida = '/home/fisica/bandas/banda13/ene_01_reducida_mes'  # Directorio para imágenes redimensionadas

# Crear directorio reducido si no existe
if not os.path.exists(d_reducida):
    os.makedirs(d_reducida)

In [9]:
# =====================================================
# FUNCIONES AUXILIARES
# =====================================================

def resize_npy_images(d_original, d_reducida, new_size=(480, 480)):
    """Redimensiona imágenes .npy al tamaño especificado"""
    print(f"Redimensionando imágenes en {d_original}...")
    
    if not os.path.exists(d_reducida):
        os.makedirs(d_reducida)
    
    processed_count = 0
    for filename in os.listdir(d_original):
        if filename.lower().endswith('.npy'):
            img_path = os.path.join(d_original, filename)
            img_array = np.load(img_path)
            
            # Redimensionar usando PIL
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            
            # Guardar imagen redimensionada
            output_path = os.path.join(d_reducida, filename)
            np.save(output_path, img_resized)
            processed_count += 1
    
    print(f"Procesadas {processed_count} imágenes")
    return processed_count

def mostrar_resultados(X, y, y_pred, n=5, cmap='viridis',
                       titulo_general="Comparación de Resultados",
                       xlabel="Eje X (pixeles)", ylabel="Eje Y (pixeles)"):
    """
    Muestra comparaciones entre imágenes de entrada, reales y predichas.
    """
    n = min(n, len(X))
    
    fig, axs = plt.subplots(n, 3, figsize=(12, 3 * n))
    fig.suptitle(titulo_general, fontsize=16, y=1.02)
    
    for i in range(n):
        # Entrada
        axs[i, 0].imshow(X[i, ..., 0], cmap=cmap)
        axs[i, 0].set_title(f'Entrada t={i}')
        axs[i, 0].set_xlabel(xlabel)
        axs[i, 0].set_ylabel(ylabel)
        
        # Real
        axs[i, 1].imshow(y[i, ..., 0], cmap=cmap)
        axs[i, 1].set_title(f'Real t+1={i+1}')
        axs[i, 1].set_xlabel(xlabel)
        axs[i, 1].set_ylabel(ylabel)
        
        # Predicción
        axs[i, 2].imshow(y_pred[i, ..., 0], cmap=cmap)
        axs[i, 2].set_title(f'Predicción t+1={i+1}')
        axs[i, 2].set_xlabel(xlabel)
        axs[i, 2].set_ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()


In [10]:
# =====================================================
# PREPROCESAMIENTO DE DATOS
# =====================================================

print("=" * 50)
print("PREPROCESAMIENTO DE IMÁGENES")
print("=" * 50)

# Redimensionar imágenes (solo si es necesario)
print("Verificando si necesitas redimensionar imágenes...")
existing_files = len([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
if existing_files == 0:
    processed_count = resize_npy_images(d_original, d_reducida, new_size=(480, 480))
    print(f"✅ Redimensionadas {processed_count} imágenes")
else:
    print(f"✅ Ya existen {existing_files} imágenes redimensionadas en {d_reducida}")

# Cargar las imágenes redimensionadas
file_list = sorted([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
print(f"\nCargando {len(file_list)} imágenes desde {d_reducida}...")

images = []
for i, filename in enumerate(file_list):
    img_path = os.path.join(d_reducida, filename)
    img_array = np.load(img_path)
    images.append(img_array)
    
    if (i + 1) % 100 == 0:  # Mostrar progreso cada 100 imágenes
        print(f"  Cargadas {i + 1}/{len(file_list)} imágenes...")

print(f"✅ Carga completada: {len(images)} imágenes")

# Normalización
print("\nNormalizando imágenes...")
images = [(img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8) for img in images]
images = [np.clip(img, 0, 1) for img in images]
print("✅ Normalización completada")

# Construir secuencias temporales
print("\nConstruyendo secuencias temporales...")
timesteps = 3

X_base = np.array(images)[..., np.newaxis]  # (num_imgs, H, W, C)
y_base = X_base.copy()

N_seq = X_base.shape[0] - timesteps
X_seq = np.array([X_base[i:i+timesteps] for i in range(N_seq)])  # (N_seq, timesteps, H, W, C)
y_seq = y_base[timesteps:]  # (N_seq, H, W, C)

print(f"✅ Secuencias creadas:")
print(f"   - Forma de X_seq: {X_seq.shape}")  # (N_seq, timesteps, H, W, C)
print(f"   - Forma de y_seq: {y_seq.shape}")  # (N_seq, H, W, C)
print(f"   - Número de secuencias: {N_seq}")


PREPROCESAMIENTO DE IMÁGENES
Verificando si necesitas redimensionar imágenes...
✅ Ya existen 741 imágenes redimensionadas en /home/fisica/bandas/banda13/ene_01_reducida_mes

Cargando 741 imágenes desde /home/fisica/bandas/banda13/ene_01_reducida_mes...
  Cargadas 100/741 imágenes...
  Cargadas 200/741 imágenes...
  Cargadas 300/741 imágenes...
  Cargadas 400/741 imágenes...
  Cargadas 500/741 imágenes...
  Cargadas 600/741 imágenes...
  Cargadas 700/741 imágenes...
✅ Carga completada: 741 imágenes

Normalizando imágenes...
✅ Normalización completada

Construyendo secuencias temporales...
✅ Secuencias creadas:
   - Forma de X_seq: (738, 3, 480, 480, 1)
   - Forma de y_seq: (738, 480, 480, 1)
   - Número de secuencias: 738


In [ ]:
# =====================================================
# ENTRENAMIENTO DEL MODELO
# =====================================================

print("\n" + "=" * 50)
print("CONFIGURACIÓN DEL MODELO")
print("=" * 50)

# Parámetros del modelo
H, W, C = X_seq.shape[2], X_seq.shape[3], X_seq.shape[4]
semilla = 42
tf.random.set_seed(semilla)

# Crear modelo
model14 = build_simplified_convLSTM(timesteps, H, W, C)

# Compilar modelo
initial_lr = 1e-3
optimizer = Adam(learning_rate=initial_lr)

model14.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[RootMeanSquaredError(), MeanAbsolutePercentageError()]
)

print("✅ Modelo compilado:")
model14.summary()


CONFIGURACIÓN DEL MODELO
✅ Modelo compilado:


Model: "ConvLSTM_simplified"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 3, 480,    │          0 │ -                 │
│ (InputLayer)        │ 480, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv3x3      │ (None, 3, 480,    │      9,856 │ input_sequence[0… │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv5x5      │ (None, 3, 480,    │     27,264 │ input_sequence[0… │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_6  │ (None, 3, 480,    │         64 │ block1_conv3x3[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_7  │ (None, 3, 480,    │         64 │ block1_conv5x5[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 3, 480,    │          0 │ activation_7[0][… │
│                     │ 480, 16)          │            │ activation_8[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_merged       │ (None, 3, 480,    │          0 │ add_4[0][0]       │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 3, 480,    │          0 │ block1_merged[0]… │
│                     │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv3x3      │ (None, 3, 480,    │     18,496 │ dropout_4[0][0]   │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv5x5      │ (None, 3, 480,    │     51,264 │ dropout_4[0][0]   │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_8  │ (None, 3, 480,    │         64 │ block2_conv3x3[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_9  │ (None, 3, 480,    │         64 │ block2_conv5x5[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 3, 480,    │          0 │ activation_9[0][

 Total params: 344,273 (1.31 MB)

 Trainable params: 344,017 (1.31 MB)

 Non-trainable params: 256 (1.00 KB)

In [13]:
# =====================================================
# CALLBACKS Y CONFIGURACIÓN DE ENTRENAMIENTO
# =====================================================

def cosine_decay_schedule(epoch, lr):
    """Learning rate schedule mejorado"""
    epochs_total = 50
    min_lr = 1e-5
    
    if epoch < 10:
        return lr
    else:
        cosine_decay = 0.5 * (1 + np.cos(np.pi * (epoch - 10) / (epochs_total - 10)))
        new_lr = min_lr + (initial_lr - min_lr) * cosine_decay
        return new_lr

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath="modelo_simplificado_best.h5",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    LearningRateScheduler(cosine_decay_schedule, verbose=1)
]


In [14]:
# =====================================================
# DIVISIÓN DE DATOS
# =====================================================

print("\n" + "=" * 50)
print("DIVISIÓN DE DATOS")
print("=" * 50)

porc_validacion = 0.2
split_index = int(len(X_seq) * (1 - porc_validacion))

X_train, X_val = X_seq[:split_index], X_seq[split_index:]
y_train, y_val = y_seq[:split_index], y_seq[split_index:]

print(f"✅ Datos divididos:")
print(f"   - Entrenamiento: {len(X_train)} muestras")
print(f"   - Validación: {len(X_val)} muestras")
print(f"   - Proporción: {len(X_train)/len(X_seq)*100:.1f}% entrenamiento, {len(X_val)/len(X_seq)*100:.1f}% validación")



DIVISIÓN DE DATOS
✅ Datos divididos:
   - Entrenamiento: 590 muestras
   - Validación: 148 muestras
   - Proporción: 79.9% entrenamiento, 20.1% validación


In [ ]:
# =====================================================
# ENTRENAMIENTO
# =====================================================

print("\n" + "=" * 50)
print("INICIANDO ENTRENAMIENTO")
print("=" * 50)

print(f"Parámetros de entrenamiento:")
print(f"   - Épocas: 50")
print(f"   - Batch size: 4")
print(f"   - Learning rate inicial: {initial_lr}")
print(f"   - Tamaño de entrada: {X_train.shape[1:]}")
print(f"   - Directorio de trabajo: {os.getcwd()}")

history = model14.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=4,
    callbacks=callbacks,
    verbose=1
)

print("✅ ¡Entrenamiento completado!")



INICIANDO ENTRENAMIENTO
Parámetros de entrenamiento:
   - Épocas: 50
   - Batch size: 4
   - Learning rate inicial: 0.001
   - Tamaño de entrada: (3, 480, 480, 1)
   - Directorio de trabajo: /home/fisica/monografia_esp_cd/Modelos


2025-10-20 20:30:01.495608: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1631232000 exceeds 10% of free system memory.



Epoch 1: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 1/50


2025-10-20 20:30:03.690311: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 543744000 exceeds 10% of free system memory.
2025-10-20 20:30:31.705597: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.
2025-10-20 20:30:33.817082: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.
2025-10-20 20:30:37.320766: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.


  1/148 ━━━━━━━━━━━━━━━━━━━━ 7:34:56 186s/step - loss: 1.0849 - mean_absolute_percentage_error: 1606.8899 - root_mean_squared_error: 1.0341